In [1]:
# This code classifies mammal sounds in audio files with amphibians in them.

import librosa
import numpy as np
import pandas as pd
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [2]:
# Lists the types of amphibians

amphibians = []

taxonomy_df = pd.read_csv("taxonomy.csv")

for row in taxonomy_df.itertuples():
    amphibians += (row.class_name == 'Amphibia')*[row.primary_label]

In [3]:
len(amphibians) # There are 35 amphibian species.

35

In [4]:
train_soundscapes_labels_df = pd.read_csv("train_soundscapes_labels.csv")

train_soundscapes_reduced_df = pd.DataFrame({
    "filename": train_soundscapes_labels_df.iloc[:, :2].astype(str).agg(''.join, axis=1),
    "primary_label": train_soundscapes_labels_df.iloc[:, 3]
})

In [5]:
train_soundscapes_reduced_df.head()

,filename,primary_label
0,BC2026_Train_0039_S22_20211231_201500.ogg00:00:00,22961;23158;24321;517063;65380
1,BC2026_Train_0039_S22_20211231_201500.ogg00:00:05,22961;23158;24321;517063;65380
2,BC2026_Train_0039_S22_20211231_201500.ogg00:00:10,22961;23158;24321;517063;65380
3,BC2026_Train_0039_S22_20211231_201500.ogg00:00:15,22961;23158;24321;517063;65380
4,BC2026_Train_0039_S22_20211231_201500.ogg00:00:20,22961;23158;24321;517063;65380


In [6]:
# Lists all files in train_soundscapes_reduced_df which contain amphibians.
# Organizes all amphibians in these files.

train_soundscapes_amphibians_df = pd.DataFrame(columns = ['filename', 'primary_label'])

for entry in train_soundscapes_reduced_df.itertuples():
    found_amphibian = False
    primary_labels = entry.primary_label.split(';')
    row_amphibians = ''
    
    for label in primary_labels:
        if label in amphibians:
            if found_amphibian:
                row_amphibians += ';'
            found_amphibian = True
            row_amphibians += label
    
    if found_amphibian:
        new_row = pd.DataFrame({'filename': [entry.filename], 'primary_label': [row_amphibians]})
        train_soundscapes_amphibians_df = pd.concat([train_soundscapes_amphibians_df, new_row], ignore_index=True)

In [7]:
train_soundscapes_amphibians_df.head()

,filename,primary_label
0,BC2026_Train_0039_S22_20211231_201500.ogg00:00:00,22961;23158;24321;517063;65380
1,BC2026_Train_0039_S22_20211231_201500.ogg00:00:05,22961;23158;24321;517063;65380
2,BC2026_Train_0039_S22_20211231_201500.ogg00:00:10,22961;23158;24321;517063;65380
3,BC2026_Train_0039_S22_20211231_201500.ogg00:00:15,22961;23158;24321;517063;65380
4,BC2026_Train_0039_S22_20211231_201500.ogg00:00:20,22961;23158;24321;517063;65380


In [8]:
# Unlike with mammals, it is actually very common to have a sound file with multiple amphibian species.
# 966 out of 1132 files in train_soundscapes_amphibians_df have multiple amphibians.

multiple_count = 0

for entry in train_soundscapes_amphibians_df.itertuples():
    if ';' in entry.primary_label:
        multiple_count += 1

print(multiple_count)

966


In [9]:
len(train_soundscapes_amphibians_df)

1132

In [10]:
# In animal_class_classifier.ipynb, we created train_audio_df.csv.
# Each row in this csv corresponds to an audio file in the train_audio folder along with its class.
# Recall that each file in the train_audio folder has exactly one animal in it.
# This loop identifies all amphibian files in train_audio_df and writes their primary labels.
# Note that the primary label is just the part of the filename that occurs before the '/'.

train_audio_df = pd.read_csv('train_audio_df.csv')

amphibian_audio_df = pd.DataFrame(columns = ['filename', 'primary_label'])

for entry in train_audio_df.itertuples():
    if entry.primary_label == 'Amphibia':
        new_row = pd.DataFrame({'filename': [entry.filename], 'primary_label': [entry.filename.split('/')[0]]})
        amphibian_audio_df = pd.concat([amphibian_audio_df, new_row], ignore_index=True)

In [11]:
len(amphibian_audio_df) # 422 files

422

In [12]:
amphibian_audio_df.head()

,filename,primary_label
0,22961/iNat846512.ogg,22961
1,22961/iNat1662496.ogg,22961
2,22961/iNat512578.ogg,22961
3,22961/XC1062603.ogg,22961
4,22961/iNat1657948.ogg,22961


In [13]:
train_soundscapes_amphibians_df.to_csv('train_soundscapes_amphibians_df.csv', index=False)
amphibian_audio_df.to_csv('amphibian_audio_df.csv', index=False)

In [14]:
# We count the number of files in each class.

counter = {}
for amphibian in amphibians:
    counter[amphibian] = 0

for label in train_soundscapes_amphibians_df.primary_label:
    labels = label.split(';')
    for primary_label in labels:
        counter[primary_label] += 1

for label in amphibian_audio_df.primary_label:
    counter[str(label)] += 1

print(counter)

{'1176823': 12, '1491113': 158, '1595929': 5, '22930': 6, '22956': 31, '22961': 78, '22967': 318, '22973': 483, '22983': 10, '22985': 6, '23150': 1, '23154': 5, '23158': 374, '23176': 4, '23724': 1, '24279': 387, '24285': 19, '24287': 11, '24321': 346, '25073': 24, '25092': 70, '25214': 3, '326272': 71, '476521': 2, '517063': 626, '555123': 3, '555145': 9, '555146': 436, '64898': 3, '65377': 60, '65380': 689, '66971': 303, '67107': 31, '67252': 9, '70711': 2}


In [15]:
common_amphibians = []
rare_amphibians = []

for amphibian in amphibians:
    if counter[amphibian] < 35:
        rare_amphibians += [amphibian]
    else:
        common_amphibians += [amphibian]

In [16]:
print(common_amphibians, len(common_amphibians)) # There are 14 common species and 21 rare species.
print(rare_amphibians, len(rare_amphibians))

['1491113', '22961', '22967', '22973', '23158', '24279', '24321', '25092', '326272', '517063', '555146', '65377', '65380', '66971'] 14
['1176823', '1595929', '22930', '22956', '22983', '22985', '23150', '23154', '23176', '23724', '24285', '24287', '25073', '25214', '476521', '555123', '555145', '64898', '67107', '67252', '70711'] 21


In [17]:
train_soundscapes_common_amphibians_df = pd.DataFrame(columns = ['filename', 'primary_label'])

for entry in train_soundscapes_amphibians_df.itertuples():
    found_common_amphibian = False
    primary_labels = entry.primary_label.split(';')
    row_common_amphibians = ''
    
    for label in primary_labels:
        if label in amphibians:
            if found_common_amphibian:
                row_common_amphibians += ';'
            found_common_amphibian = True
            row_common_amphibians += label
    
    if found_common_amphibian:
        new_row = pd.DataFrame({'filename': [entry.filename], 'primary_label': [row_common_amphibians]})
        train_soundscapes_common_amphibians_df = pd.concat([train_soundscapes_common_amphibians_df, new_row], ignore_index=True)

In [18]:
# All of the files in train_soundscapes_amphibians had a common amphibian.

print(len(train_soundscapes_amphibians_df), len(train_soundscapes_common_amphibians_df))

1132 1132


In [19]:
train_soundscapes_common_amphibians_df.head()

,filename,primary_label
0,BC2026_Train_0039_S22_20211231_201500.ogg00:00:00,22961;23158;24321;517063;65380
1,BC2026_Train_0039_S22_20211231_201500.ogg00:00:05,22961;23158;24321;517063;65380
2,BC2026_Train_0039_S22_20211231_201500.ogg00:00:10,22961;23158;24321;517063;65380
3,BC2026_Train_0039_S22_20211231_201500.ogg00:00:15,22961;23158;24321;517063;65380
4,BC2026_Train_0039_S22_20211231_201500.ogg00:00:20,22961;23158;24321;517063;65380


In [20]:
common_amphibian_audio_df = pd.DataFrame(columns = ['filename', 'primary_label'])

for entry in amphibian_audio_df.itertuples():
    if str(entry.primary_label) in common_amphibians:
        new_row = pd.DataFrame({'filename': [entry.filename], 'primary_label': [str(entry.primary_label)]})
        common_amphibian_audio_df = pd.concat([common_amphibian_audio_df, new_row], ignore_index=True)

In [21]:
print(len(amphibian_audio_df), len(common_amphibian_audio_df))

422 271


In [22]:
common_amphibian_audio_df.head()

,filename,primary_label
0,22961/iNat846512.ogg,22961
1,22961/iNat1662496.ogg,22961
2,22961/iNat512578.ogg,22961
3,22961/XC1062603.ogg,22961
4,22961/iNat1657948.ogg,22961


In [23]:
# We now have all the audio files (train_soundscapes_common_amphibians_df and common_amphibian_audio_df).

NUM_CLASSES = len(common_amphibians)
SR = 32000
N_MELS = 128
X_multi = []
y_multi = []

# This loop converts each audio file in train_soundscapes_amphibians_df into a mel spectrogram.

for row in train_soundscapes_common_amphibians_df.itertuples():
    filename = row.filename
    labels = row.primary_label.split(';')
    file = filename[:-8]
    start = int(filename[-2:])
    
    audio, _ = librosa.load(
        f"train_soundscapes/{file}",
        sr=SR,
        offset=start,
        duration=5
    )

# Converts the audio to a mel spectrogram, and normalizes it to the range [0, 1].

    mel = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=N_MELS)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min())

    X_multi.append(mel_db[..., np.newaxis])
    
    # 🔥 MULTI-LABEL TARGET
    label_vector = [0] * NUM_CLASSES
    for i, amphibian in enumerate(common_amphibians):
        if amphibian in labels:
            label_vector[i] = 1

    y_multi.append(label_vector)
    
X_multi = np.array(X_multi, dtype=np.float32)
y_multi = np.array(y_multi, dtype=np.float32)

# X is the list of spectrograms, and y is the list of multi-label vectors.

In [24]:
len(y_multi)

1132

In [27]:
# We repeat the same process for the audio files in common_amphibian_audio_df.
# Because these files are longer than 5 seconds, we split them into 5 second chunks.
# In animal_class_classifier.ipynb, we had too many files and took random 5 second intervals from each of them.
# Because we have fewer files in this version, we may use every interval.

X_single = []
y_single = []

EXPECTED_FRAMES = int(np.ceil((SR * 5) / 512))

for row in common_amphibian_audio_df.itertuples():    
    filename = row.filename
    label = row.primary_label
    
    audio, _ = librosa.load(
        f"train_audio/{filename}",
        sr=SR
    )

    chunk_length = 5 * SR
    
    # In animal_class_classifier, we ensured that all of the ogg files were at least five seconds long.
    audio_clips = []
    for i in range(int(len(audio)/chunk_length)):
        audio_clips += [audio[i*chunk_length:(i + 1)*chunk_length]]

    # Converts the audio files in audio_clips to a mel spectrogram, and normalizes it to the range [0, 1].

    for audio_clip in audio_clips:
        mel = librosa.feature.melspectrogram(
            y=audio_clip,
            sr=SR,
            n_mels=N_MELS
        )

        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min())

        # Pad / trim
        if mel_db.shape[1] < EXPECTED_FRAMES:
            mel_db = np.pad(mel_db, ((0, 0), (0, EXPECTED_FRAMES - mel_db.shape[1])))
        else:
            mel_db = mel_db[:, :EXPECTED_FRAMES]

        X_single.append(mel_db[..., np.newaxis])

        label_vector = [0] * NUM_CLASSES
        for j, amphibian in enumerate(common_amphibians):
            if amphibian == label:
                label_vector[j] = 1

        y_single.append(label_vector)

X_single = np.array(X_single, dtype=np.float32)
y_single = np.array(y_single, dtype=np.float32)

In [28]:
print(len(y_single)) # We have 1132 multi-class files and 1483 single class files.

1483


In [31]:
X = np.concatenate([X_multi, X_single], axis=0)
y = np.concatenate([y_multi, y_single], axis=0)

In [32]:
# We shuffle the elements of X and y so that we do not over-sample certain types of data.

indices = np.arange(len(X))
np.random.shuffle(indices)

X = X[indices]
y = y[indices]

np.save("X_amphibian.npy", X)
np.save("y_amphibian.npy", y)

In [33]:
# SpecAugment introduces some light noise into our training data to make it more robust.

def spec_augment(mel, p=0.3):
    if np.random.rand() > p:
        return mel

    mel = mel.copy()
    n_mels, n_frames = mel.shape

    # light time mask
    t = np.random.randint(5, min(20, n_frames // 5))
    t0 = np.random.randint(0, n_frames - t)
    mel[:, t0:t0+t] = mel.mean()

    # light freq mask
    f = np.random.randint(3, min(10, n_mels // 5))
    f0 = np.random.randint(0, n_mels - f)
    mel[f0:f0+f, :] = mel.mean()

    return mel

In [34]:
def preprocess(mel):
    if np.random.rand() < 0.3:
        mel = spec_augment(mel)
    return mel

In [35]:
# Performs a train-test-validation split.

# Step 1: Train vs temp (val + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,          # 70% train, 30% temp
    random_state=42,
    stratify=y.argmax(axis=1)
)

# Step 2: Split temp into val + test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,          # 15% val, 15% test
    random_state=42,
    stratify=y_temp.argmax(axis=1)
)

X_train = preprocess(X_train)

In [36]:
# We oversample rare classes so that we do not ignore them.

class_indices = {
    c: np.where(y_train[:, c] == 1)[0]
    for c in range(NUM_CLASSES)
}

max_count = max(len(idxs) for idxs in class_indices.values())

X_balanced = []
y_balanced = []

for c in range(NUM_CLASSES):
    idxs = class_indices[c]

    # sample WITH replacement to match max_count
    sampled_idxs = np.random.choice(idxs, size=max_count, replace=True)

    X_balanced.append(X_train[sampled_idxs])
    y_balanced.append(y_train[sampled_idxs])

X_balanced = np.concatenate(X_balanced, axis=0)
y_balanced = np.concatenate(y_balanced, axis=0)

In [37]:
perm = np.random.permutation(len(X_balanced))
X_balanced = X_balanced[perm]
y_balanced = y_balanced[perm]

In [38]:
np.save("X_train.npy", X_train)
np.save("X_val.npy", X_val)
np.save("X_test.npy", X_test)

In [39]:
def data_generator(X, y, batch_size=32):
    while True:
        idx = np.random.choice(len(X), batch_size)
        X_batch = X[idx]
        y_batch = y[idx]
        yield X_batch, y_batch

In [40]:
len(y_train), len(y_val), len(y_test)

(1830, 392, 393)

In [41]:
import tensorflow as tf
from tensorflow.keras import layers, models

input_shape = X.shape[1:]  # (128, frames, 1)

def build_model(input_shape=(128, 313, 1), num_classes=14):
    inputs = tf.keras.Input(shape=input_shape)

    # Block 1
    x = layers.Conv2D(32, (3,3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)

    # Block 2
    x = layers.Conv2D(64, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)

    # Block 3
    x = layers.Conv2D(128, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)

    # Block 4 (optional but helpful)
    x = layers.Conv2D(256, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.GlobalAveragePooling2D()(x)

    # Dense head
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(num_classes, activation='sigmoid')(x)

    return models.Model(inputs, outputs)

model = build_model(input_shape=(128, 313, 1), num_classes=14)

In [42]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.05),
    metrics=[
        tf.keras.metrics.AUC(curve='ROC', multi_label=True, name='auc'),
        tf.keras.metrics.AUC(curve='PR', multi_label=True, name='pr_auc'),
        tf.keras.metrics.BinaryAccuracy(threshold=0.5)
    ]
)

In [43]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_pr_auc',
        patience=8,
        min_delta=0.001,
        mode='max',
        restore_best_weights=True,
        verbose=1
    ),
    
    tf.keras.callbacks.ReduceLROnPlateau( # This block reduces the learning rate near the solution
        monitor='val_loss',
        factor=0.5,
        patience=3,
        cooldown=1,
        mode='max',
        min_lr=1e-6,
        verbose=1
    ),
    
    tf.keras.callbacks.ModelCheckpoint(
        'best_model.keras',
        monitor='val_pr_auc',
        mode='max',
        save_best_only=True,
        verbose=1
    )
]

In [44]:
model.fit(
    X_balanced, y_balanced,
    batch_size=32,
    shuffle=True,
    validation_data=(X_val, y_val),
    epochs=30,
    callbacks=callbacks
)

Epoch 1/30
230/230 ━━━━━━━━━━━━━━━━━━━━ 0s 514ms/step - auc: 0.7686 - binary_accuracy: 0.8080 - loss: 0.4562 - pr_auc: 0.5065
Epoch 1: val_pr_auc improved from None to 0.26272, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
230/230 ━━━━━━━━━━━━━━━━━━━━ 122s 521ms/step - auc: 0.8476 - binary_accuracy: 0.8511 - loss: 0.3958 - pr_auc: 0.6169 - val_auc: 0.6359 - val_binary_accuracy: 0.8489 - val_loss: 0.6757 - val_pr_auc: 0.2627 - learning_rate: 3.0000e-04
Epoch 2/30
230/230 ━━━━━━━━━━━━━━━━━━━━ 0s 492ms/step - auc: 0.9281 - binary_accuracy: 0.9020 - loss: 0.3143 - pr_auc: 0.7852
Epoch 2: val_pr_auc improved from 0.26272 to 0.37704, saving model to best_model.keras

Epoch 2: finished saving model to best_model.keras
230/230 ━━━━━━━━━━━━━━━━━━━━ 114s 497ms/step - auc: 0.9369 - binary_accuracy: 0.9094 - loss: 0.3025 - pr_auc: 0.8061 - val_auc: 0.7735 - val_binary_accuracy: 0.8495 - val_loss: 0.5944 - val_pr_auc: 0.3770 - learning_rate: 3.0000e-04
Epoch 3

In [45]:
y_pred = model.predict(X_val)

print(y_pred.mean(axis=0))
print(y_pred.std(axis=0))

13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 105ms/step
[0.09064494 0.06582962 0.1733319  0.27312118 0.2027787  0.22324778
 0.16578113 0.10408847 0.10225461 0.27048418 0.17308941 0.12438495
 0.29886916 0.14717536]
[0.2187993  0.16781805 0.279553   0.36016876 0.33653736 0.33672768
 0.30265868 0.192892   0.20225227 0.3556635  0.30200875 0.21964076
 0.39521757 0.27967322]


In [87]:
from sklearn.metrics import f1_score

thresholds = []

for i in range(NUM_CLASSES):
    best_t = 0.5
    best_f1 = 0

    for t in np.linspace(0.05, 0.9, 50):
        preds = (y_pred[:, i] > t).astype(int)
        f1 = f1_score(y_val[:, i], preds)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    thresholds.append(best_t)

print(thresholds)

[np.float64(0.6571428571428571), np.float64(0.2755102040816326), np.float64(0.4836734693877551), np.float64(0.4316326530612245), np.float64(0.29285714285714287), np.float64(0.4663265306122449), np.float64(0.6224489795918368), np.float64(0.36224489795918363), np.float64(0.639795918367347), np.float64(0.4316326530612245), np.float64(0.6224489795918368), np.float64(0.34489795918367344), np.float64(0.5877551020408164), np.float64(0.553061224489796)]


In [88]:
for i in range(NUM_CLASSES):
    print(i, y_pred[:, i].min(), y_pred[:, i].max())

0 0.0016725053 0.9779789
1 0.00088895945 0.96235204
2 0.0010508477 0.98812014
3 0.0045569567 0.99731904
4 0.0014701377 0.9981861
5 0.0016406822 0.9993307
6 0.00217066 0.9879815
7 0.00035109732 0.9503229
8 0.0017554744 0.9813489
9 0.0020855814 0.99679536
10 0.0027621954 0.99480695
11 0.00044879632 0.9770668
12 0.002670448 0.99626887
13 0.0016107302 0.9819179


In [89]:
y_pred_binary = np.zeros_like(y_pred)

for i in range(NUM_CLASSES):
    y_pred_binary[:, i] = (y_pred[:, i] > thresholds[i]).astype(int)

In [90]:
from sklearn.metrics import classification_report

print(classification_report(y_val, y_pred_binary))

              precision    recall  f1-score   support

           0       1.00      0.96      0.98        24
           1       1.00      1.00      1.00        16
           2       0.98      1.00      0.99        50
           3       0.98      0.95      0.97       108
           4       0.94      0.99      0.96        80
           5       0.98      0.98      0.98        88
           6       0.96      0.98      0.97        48
           7       1.00      1.00      1.00        22
           8       0.94      0.91      0.92        32
           9       0.91      0.98      0.94        85
          10       0.98      0.91      0.95        70
          11       0.85      0.98      0.91        42
          12       0.96      0.94      0.95       104
          13       1.00      0.93      0.97        45

   micro avg       0.96      0.96      0.96       814
   macro avg       0.96      0.96      0.96       814
weighted avg       0.96      0.96      0.96       814
 samples avg       0.92   

/Users/noah/tf_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [46]:
model.save("best_common_amphibian_model.keras")